In [1]:
import os, sys, json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT  = r'C:\Users\chiku\OneDrive\Documents\ecom_dynamic_pricing_optimization'
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR    = os.path.join(PROJECT_ROOT, 'models')

# Load all results
with open(os.path.join(PROCESSED_DIR, 'baseline_results.json')) as f:
    baseline = json.load(f)
with open(os.path.join(PROCESSED_DIR, 'dcn_results.json')) as f:
    dcn = json.load(f)
with open(os.path.join(PROCESSED_DIR, 'hpo_results.json')) as f:
    hpo = json.load(f)
with open(os.path.join(PROCESSED_DIR, 'dml_results.json')) as f:
    dml = json.load(f)
with open(os.path.join(PROCESSED_DIR, 'optimizer_summary.json')) as f:
    optimizer = json.load(f)

# Load data
train = pd.read_parquet(os.path.join(PROCESSED_DIR, 'train_with_cate.parquet'))
recommendations = pd.read_parquet(os.path.join(PROCESSED_DIR, 'pricing_recommendations.parquet'))

print('All results loaded ✅')
print(f'\nResults available:')
print(f'  Baseline (Ridge):    MAPE={baseline["ridge_mape"]:.4f}, R²={baseline["ridge_r2"]:.4f}')
print(f'  DCN:                 MAPE={dcn["dcn_mape"]:.4f}, R²={dcn["dcn_r2"]:.4f}')
print(f'  LightGBM (tuned):    MAPE={hpo["best_mape"]:.4f}, R²={hpo["best_r2"]:.4f}')
print(f'  DML elasticity:      ATE={dml["ate"]:.4f}, bias={dml["endogeneity_bias"]:.4f}')
print(f'  Revenue lift:        {optimizer["overall_revenue_lift_pct"]:.1%}')

All results loaded ✅

Results available:
  Baseline (Ridge):    MAPE=0.4292, R²=0.0176
  DCN:                 MAPE=0.4243, R²=0.0319
  LightGBM (tuned):    MAPE=0.4177, R²=0.0553
  DML elasticity:      ATE=-0.0829, bias=0.0874
  Revenue lift:        30.0%


In [2]:
# ─── Final Model Comparison Chart ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('E-Commerce Dynamic Pricing — Final Evaluation Report',
             fontsize=14, fontweight='bold', y=1.01)

models      = ['Ridge\n(baseline)', 'DCN\n(deep learning)', 'LightGBM\n(tuned)']
mape_scores = [baseline['ridge_mape'], dcn['dcn_mape'], hpo['best_mape']]
r2_scores   = [baseline['ridge_r2'],   dcn['dcn_r2'],   hpo['best_r2']]
colors      = ['#94A3B8', '#60A5FA', '#2563EB']

# ── Plot 1: MAPE comparison ──
bars = axes[0,0].bar(models, mape_scores, color=colors, alpha=0.85, edgecolor='white')
axes[0,0].set_ylabel('MAPE (lower is better)')
axes[0,0].set_title('Demand Prediction — MAPE')
axes[0,0].set_ylim(0.40, 0.44)
for bar, val in zip(bars, mape_scores):
    axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
                   f'{val:.4f}', ha='center', va='bottom', fontsize=10)

# ── Plot 2: R² comparison ──
bars = axes[0,1].bar(models, r2_scores, color=colors, alpha=0.85, edgecolor='white')
axes[0,1].set_ylabel('R² (higher is better)')
axes[0,1].set_title('Demand Prediction — R²')
for bar, val in zip(bars, r2_scores):
    axes[0,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                   f'{val:.4f}', ha='center', va='bottom', fontsize=10)

# ── Plot 3: Elasticity comparison — OLS bias vs DML ──
estimators  = ['Naive OLS\n(biased)', 'OLS +\ncontrols\n(biased)', 'DML\n(causal)']
elasticities = [
    baseline['naive_ols_elasticity'],
    baseline['ols_controls_elasticity'],
    dml['ate'],
]
bar_colors = ['#EF4444', '#F97316', '#16A34A']
bars = axes[1,0].bar(estimators, elasticities, color=bar_colors, alpha=0.85, edgecolor='white')
axes[1,0].axhline(0, color='black', lw=0.8, linestyle='--')
axes[1,0].set_ylabel('Price Elasticity Estimate')
axes[1,0].set_title('Causal Estimation — Endogeneity Correction\n(OLS wrong sign → DML correct)')
for bar, val in zip(bars, elasticities):
    axes[1,0].text(bar.get_x() + bar.get_width()/2,
                   bar.get_height() + (0.002 if val >= 0 else -0.006),
                   f'{val:.4f}', ha='center', va='bottom', fontsize=10)

# ── Plot 4: Revenue lift by department ──
dept_revenue = (
    recommendations.groupby('department')
    .agg(current=('revenue_current', 'sum'),
         optimal=('revenue_optimal', 'sum'))
    .assign(lift=lambda x: (x['optimal'] / x['current'] - 1) * 100)
    .sort_values('lift', ascending=True)
    .head(10)
)
axes[1,1].barh(dept_revenue.index, dept_revenue['lift'],
               color='#16A34A', alpha=0.8, edgecolor='white')
axes[1,1].axvline(0, color='black', lw=0.8)
axes[1,1].set_xlabel('Revenue Lift (%)')
axes[1,1].set_title('Pricing Optimizer — Revenue Lift by Department')

plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'final_evaluation_report.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Final evaluation chart saved ✅')

Final evaluation chart saved ✅


In [3]:
# ─── Final Project Summary ────────────────────────────────────────────────────
print('=' * 60)
print('E-COMMERCE DYNAMIC PRICING — PROJECT COMPLETE')
print('=' * 60)

print(f'\n📊 DATASET')
print(f'   Transactions:          32,434,489 rows')
print(f'   Products:              49,677 SKUs')
print(f'   Departments:           21')
print(f'   Features engineered:   19')

print(f'\n🤖 DEMAND MODELS')
print(f'   {"Model":<25} {"MAPE":>8} {"R²":>8}')
print(f'   {"-"*43}')
print(f'   {"Ridge (baseline)":<25} {baseline["ridge_mape"]:>8.4f} {baseline["ridge_r2"]:>8.4f}')
print(f'   {"DCN (deep learning)":<25} {dcn["dcn_mape"]:>8.4f} {dcn["dcn_r2"]:>8.4f}')
print(f'   {"LightGBM (tuned)":<25} {hpo["best_mape"]:>8.4f} {hpo["best_r2"]:>8.4f}  ✅ best')

print(f'\n⚗️  CAUSAL ESTIMATION (Double ML)')
print(f'   Naive OLS elasticity:  {baseline["naive_ols_elasticity"]:>+.4f}  ← wrong sign (biased)')
print(f'   DML elasticity (ATE):  {dml["ate"]:>+.4f}  ← correct sign (causal) ✅')
print(f'   Endogeneity bias:       {dml["endogeneity_bias"]:.4f}')
print(f'   95% CI:                [{dml["ate_ci_lower"]:.4f}, {dml["ate_ci_upper"]:.4f}]')

print(f'\n💰 PRICING OPTIMIZER')
print(f'   Products priced:        {optimizer["n_products"]:,}')
print(f'   Mean price change:     {optimizer["mean_price_change_pct"]:>+.1%}')
print(f'   Revenue lift:          {optimizer["overall_revenue_lift_pct"]:>+.1%}')

print(f'\n📁 ARTIFACTS SAVED')
print(f'   models/dml_model.pkl        ← causal elasticity model')
print(f'   models/lgbm_tuned.pkl       ← best demand model')
print(f'   models/dcn_model.pt         ← deep learning model')
print(f'   data/processed/train_with_cate.parquet')
print(f'   data/processed/pricing_recommendations.parquet')
print(f'   data/processed/final_evaluation_report.png')

print(f'\n🚀 NEXT STEPS')
print(f'   1. Run FastAPI locally:  uvicorn src.api.main:app --reload')
print(f'   2. Start full stack:     docker-compose up (in docker/ folder)')
print(f'   3. Track experiments:    mlflow ui')
print(f'   4. Run tests:            pytest tests/')
print('=' * 60)
print('All 8 notebooks complete ✅')

E-COMMERCE DYNAMIC PRICING — PROJECT COMPLETE

📊 DATASET
   Transactions:          32,434,489 rows
   Products:              49,677 SKUs
   Departments:           21
   Features engineered:   19

🤖 DEMAND MODELS
   Model                         MAPE       R²
   -------------------------------------------
   Ridge (baseline)            0.4292   0.0176
   DCN (deep learning)         0.4243   0.0319
   LightGBM (tuned)            0.4177   0.0553  ✅ best

⚗️  CAUSAL ESTIMATION (Double ML)
   Naive OLS elasticity:  +0.0045  ← wrong sign (biased)
   DML elasticity (ATE):  -0.0829  ← correct sign (causal) ✅
   Endogeneity bias:       0.0874
   95% CI:                [-0.2226, 0.0568]

💰 PRICING OPTIMIZER
   Products priced:        29,094
   Mean price change:     +30.0%
   Revenue lift:          +30.0%

📁 ARTIFACTS SAVED
   models/dml_model.pkl        ← causal elasticity model
   models/lgbm_tuned.pkl       ← best demand model
   models/dcn_model.pt         ← deep learning model
   data/proce